# 06 - Optional Reward Model Scoring

This notebook scores matched reward-pair responses with an open reward model, saves scores to Drive, logs preference deltas to W&B, and publishes the score dataset to Hugging Face.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, textwrap

DEFAULT_REPO_URL = "https://github.com/ritwikraha/AutoRegressive-Bhasha.git"

def find_repo_root():
    try:
        import google.colab  # type: ignore  # noqa: F401
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass

    candidates = [
        Path.cwd(),
        Path("/content/empty-negations"),
        Path("/content/AutoRegressive-Bhasha/empty-negations"),
        Path("/content/drive/MyDrive/ocn_empty_negations"),
        Path("/content/drive/MyDrive/AutoRegressive-Bhasha/empty-negations"),
    ]
    for candidate in candidates:
        if (candidate / "src/ocn").exists():
            return candidate
    repo_url = os.environ.get("OCN_REPO_URL", DEFAULT_REPO_URL)
    target = Path("/content/AutoRegressive-Bhasha")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(target)], check=True)

    cloned_candidates = [target / "empty-negations", target]
    for candidate in cloned_candidates:
        if (candidate / "src/ocn").exists():
            return candidate

    raise FileNotFoundError(
        f"Cloned {repo_url}, but could not find src/ocn. "
        "Set OCN_REPO_URL to a repository containing empty-negations/src/ocn."
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo:", REPO_ROOT)

In [ ]:
import json
from pathlib import Path
import pandas as pd
import torch
import wandb
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from ocn.colab_utils import login_huggingface, login_wandb, make_colab_paths, publish_dataframe_to_hf, save_dataframe, utc_timestamp

paths = make_colab_paths()
config = json.loads((paths.project_root / "ocn_colab_config.json").read_text())
_ = login_huggingface("HF_WRITE_ACCESS")
EXPERIMENT_ID = "main_gemma4_qwen35"
REWARD_SCORE_RUN_ID = utc_timestamp()
MAIN_REWARD_PAIR_REPO = config.get(
    "hf_main_reward_pairs_repo",
    f"{config['hf_owner']}/ocn-empty-negations-reward-pairs-main-gemma4-qwen35",
)
MAIN_REWARD_SCORE_REPO = config.get(
    "hf_main_reward_scores_repo",
    f"{config['hf_owner']}/ocn-empty-negations-reward-scores-main-gemma4-qwen35",
)
score_config = {
    **config,
    "experiment_id": EXPERIMENT_ID,
    "reward_score_run_id": REWARD_SCORE_RUN_ID,
    "source_repo": MAIN_REWARD_PAIR_REPO,
    "output_repo": MAIN_REWARD_SCORE_REPO,
}
run = login_wandb(
    project="ocn-empty-negations",
    name=f"reward-score-{EXPERIMENT_ID}-{REWARD_SCORE_RUN_ID}",
    config=score_config,
)

In [ ]:
pairs = load_dataset(MAIN_REWARD_PAIR_REPO, split="train").to_pandas()
REWARD_MODEL_ID = "OpenAssistant/reward-model-deberta-v3-large-v2"

tokenizer = AutoTokenizer.from_pretrained(REWARD_MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(REWARD_MODEL_ID, device_map="auto")
model.eval()

def score_text(text: str) -> float:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model(**inputs)
    return float(out.logits.squeeze().detach().cpu())

pairs["reward_model_id"] = REWARD_MODEL_ID
pairs["reward_score"] = [score_text(text) for text in pairs["response"]]
pairs.head()

In [ ]:
scores_path = save_dataframe(
    pairs,
    Path(config["drive_data_root"]) / "ocn_reward_scores_main_gemma4_qwen35.csv",
)
repo_url = publish_dataframe_to_hf(
    pairs,
    repo_id=MAIN_REWARD_SCORE_REPO,
    split="train",
    private=config["hf_private"],
    commit_message=f"Publish {EXPERIMENT_ID} reward scores {REWARD_SCORE_RUN_ID}",
)

deltas = (
    pairs.pivot_table(index="pair_id", columns="variant_type", values="reward_score", aggfunc="mean")
    .assign(
        candidate_ocn_minus_plain=lambda x: x["candidate_ocn"] - x["plain_rewrite"],
    )
    .reset_index()
)
wandb.log({
    "reward_scores": wandb.Table(dataframe=pairs),
    "reward_deltas": wandb.Table(dataframe=deltas),
    "mean_candidate_ocn_minus_plain": float(deltas["candidate_ocn_minus_plain"].mean()),
})
run.finish()
print("Saved:", scores_path)
print("Published:", repo_url)
deltas.describe()